# YieldMPNN 模型展示教程

## 概述

本 notebook 用于拆解 YieldMPNN 的模型结构和推理原理，重点回答三个问题：

1. 单个分子图是如何被 `MPNN` 编码成一个 graph embedding 的？
2. 多个 reactant slot 和 product slot 又是如何合成为一个 reaction-level 表示的？
3. 为什么模型会同时输出 `mean` 和 `logvar`，以及 MC dropout 如何把不确定性拆成 epistemic / aleatoric 两部分？

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 准备一个与原仓库兼容的小 batch |
| 2 | 拆解单分子 `MPNN` 编码器 |
| 3 | 观察 message passing 与 readout 形状 |
| 4 | 组装顶层 `reactionMPNN` |
| 5 | 跑一次前向推理与训练损失 |
| 6 | 演示 MC dropout 不确定性估计 |
| 7 | 与原仓库类实现对照 |
| 8 | 总结与下一步 |

### 下面这个代码单元会做什么？

它会完成模型 notebook 的初始化：导入图神经网络依赖、注册外部源码路径，并准备后面要直接复用的 `GraphDataset / collate_reaction_graphs / MC_dropout / reactionMPNN`。
补充说明：这一格还会先设置 `MKL_THREADING_LAYER=GNU` 与 `OMP_NUM_THREADS=1`，避免后续图神经网络前向计算在某些 CPU-only PyTorch / MKL 组合下触发 OpenMP 线程层冲突。


In [ ]:
# ====== Step 0: 导入与路径 ======
from pathlib import Path
from contextlib import contextmanager
import os

# 先设置数值库线程策略，避免某些 CPU-only PyTorch / MKL 组合触发 OpenMP 符号问题。
os.environ.setdefault('MKL_THREADING_LAYER', 'GNU')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys
import numpy as np
import torch
import torch.nn as nn
import dgl

from torch.utils.data import DataLoader, Subset
from dgl.nn.pytorch import NNConv, Set2Set


def locate_project_root(start: Path | None = None, root_name: str = 'Chemical_Synthesis') -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if candidate.name == root_name:
            return candidate
    raise RuntimeError(f'Cannot locate project root named {root_name!r} from {path}')


@contextmanager
def pushd(path: Path):
    old = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


PROJECT_ROOT = locate_project_root()
SOURCE_REPO = (PROJECT_ROOT / '../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn').resolve()
sys.path.insert(0, str(SOURCE_REPO))

from dataset import GraphDataset
from util import collate_reaction_graphs, MC_dropout
from model import reactionMPNN as SourceReactionMPNN

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SOURCE_REPO =', SOURCE_REPO)

## 1. 准备一个与原仓库兼容的小 batch

### 为什么模型 notebook 直接复用原仓库数据层？

因为这一份 notebook 的重点不是再讲一次预处理，而是讲模型计算过程。为了把注意力放在 `model.py` 上，这里直接调用：

- `dataset.py` 中的 `GraphDataset`
- `util.py` 中的 `collate_reaction_graphs`

这样我们拿到的 batch 结构就和原仓库训练时完全一致。

### 下面这个代码单元会做什么？

它会从原仓库数据层中抽出一个小 batch，并打印 reactant slot、product slot 和标签张量。后面所有模型讲解都基于这一个 batch 展开。



In [ ]:
# ====== Step 1: 构造教学 batch ======
with pushd(SOURCE_REPO):
    dataset = GraphDataset(data_id=1, split_id=0)
    subset = Subset(dataset, list(range(4)))
    loader = DataLoader(subset, batch_size=4, shuffle=False, collate_fn=collate_reaction_graphs)
    batch = next(iter(loader))

rmol_max_cnt = dataset.rmol_max_cnt
pmol_max_cnt = dataset.pmol_max_cnt
rmols = batch[:rmol_max_cnt]
pmols = batch[rmol_max_cnt:rmol_max_cnt + pmol_max_cnt]
labels = batch[-1]

print('dataset length =', len(dataset))
print('rmol_max_cnt =', rmol_max_cnt)
print('pmol_max_cnt =', pmol_max_cnt)
print('batch labels =', labels.numpy())
print('reactant slot 0 node_attr shape =', tuple(rmols[0].ndata['attr'].shape))
print('reactant slot 0 edge_attr shape =', tuple(rmols[0].edata['edge_attr'].shape))
print('product slot 0 node_attr shape =', tuple(pmols[0].ndata['attr'].shape))
print('per-sample reactant occupancy mask =')
print(torch.stack([mol.batch_num_nodes() > 0 for mol in rmols]).int())

## 2. 拆解单分子 `MPNN` 编码器

### 核心思想

原仓库里的 `MPNN` 由 4 个关键组件组成：

1. `project_node_feats`
   - 把原始节点特征从 43 维投影到隐藏维度 64。
2. `NNConv`
   - 用边特征生成 message passing 时的动态权重。
3. `GRU`
   - 把多步消息传递结果串成一个递归更新过程。
4. `Set2Set + sparsify`
   - 把 node-level 表示汇聚成 graph-level embedding，再投影到 1024 维。

### 源码对应

- 文件：`model.py`
- 类：`MPNN`

### 形状主线

如果默认 `hidden_feats = 64`，那么张量流大致是：

`node_attr(43) -> hidden(64) -> concat(initial, final)=128 -> Set2Set output=256 -> graph_feats=1024`

### 下面这个代码单元会做什么？

它会把原仓库 `MPNN` 拆成一个可观察的教学版类。对应的张量流可以概括成：

$$
\mathbf{h}^{(0)} = \mathrm{MLP}(\mathbf{x}), \qquad
\mathbf{h}^{(t+1)} = \mathrm{GRU}\bigl(\mathrm{NNConv}(\mathbf{h}^{(t)}, \mathbf{e}),\ \mathbf{h}^{(t)}\bigr)
$$

下面这格代码的价值在于：它不仅能前向，还能把每一步形状记录下来。



In [ ]:
# ====== Step 2: 教学版单分子 MPNN ======
class TeachingMPNN(nn.Module):
    def __init__(
        self,
        node_in_feats: int,
        edge_in_feats: int,
        hidden_feats: int = 64,
        num_step_message_passing: int = 3,
        num_step_set2set: int = 3,
        num_layer_set2set: int = 1,
        readout_feats: int = 1024,
    ):
        super().__init__()
        self.project_node_feats = nn.Sequential(nn.Linear(node_in_feats, hidden_feats), nn.ReLU())
        self.num_step_message_passing = num_step_message_passing
        self.gnn_layer = NNConv(
            in_feats=hidden_feats,
            out_feats=hidden_feats,
            edge_func=nn.Linear(edge_in_feats, hidden_feats * hidden_feats),
            aggregator_type='sum',
        )
        self.activation = nn.ReLU()
        self.gru = nn.GRU(hidden_feats, hidden_feats)
        self.readout = Set2Set(
            input_dim=hidden_feats * 2,
            n_iters=num_step_set2set,
            n_layers=num_layer_set2set,
        )
        self.sparsify = nn.Sequential(nn.Linear(hidden_feats * 4, readout_feats), nn.PReLU())

    def forward(self, g, return_trace: bool = False):
        node_feats = g.ndata['attr']
        edge_feats = g.edata['edge_attr']

        projected = self.project_node_feats(node_feats)
        hidden_state = projected.unsqueeze(0)
        current = projected
        step_trace = []

        for step in range(self.num_step_message_passing):
            conv_out = self.activation(self.gnn_layer(g, current, edge_feats))
            gru_out, hidden_state = self.gru(conv_out.unsqueeze(0), hidden_state)
            current = gru_out.squeeze(0)
            step_trace.append(
                {
                    'step': step + 1,
                    'conv_out': tuple(conv_out.shape),
                    'gru_out': tuple(current.shape),
                }
            )

        node_aggr = torch.cat([projected, current], dim=1)
        readout = self.readout(g, node_aggr)
        graph_feats = self.sparsify(readout)

        if return_trace:
            trace = {
                'input_node_attr': tuple(node_feats.shape),
                'input_edge_attr': tuple(edge_feats.shape),
                'projected': tuple(projected.shape),
                'message_steps': step_trace,
                'node_aggr': tuple(node_aggr.shape),
                'readout': tuple(readout.shape),
                'graph_feats': tuple(graph_feats.shape),
            }
            return graph_feats, trace
        return graph_feats

## 3. 观察 message passing 与 readout 形状

### 为什么这一节值得单独看？

很多人第一次读 `model.py` 时，只能看到一个黑盒 `self.mpnn(mol)`。但真正帮助理解模型的是：

- 节点表示在每一轮 message passing 后如何变化
- `Set2Set` 为什么会把 128 维 node 表示汇聚成 256 维 graph 表示
- 最后的 `sparsify` 为什么输出 1024 维图向量

### 下面这个代码单元会做什么？

它会真正跑一次单分子编码，并打印 `projected / message_steps / node_aggr / readout / graph_feats` 的形状，让读者看到抽象模块在具体 batch 上到底生成了什么。



In [ ]:
# ====== Step 3: 观察单分子编码器的张量流 ======
node_dim = dataset.rmol_node_attr[0].shape[1]
edge_dim = dataset.rmol_edge_attr[0].shape[1]

teaching_mpnn = TeachingMPNN(node_in_feats=node_dim, edge_in_feats=edge_dim)
slot0_graph_feats, slot0_trace = teaching_mpnn(rmols[0], return_trace=True)

print('slot0 graph_feats shape =', tuple(slot0_graph_feats.shape))
print('trace summary:')
for key, value in slot0_trace.items():
    print(f'- {key}: {value}')

## 4. 组装顶层 `reactionMPNN`

### 核心思想

顶层模型做的事情其实很直接：

1. 对每个 reactant slot 分别调用同一个 `MPNN`
2. 用 `batch_num_nodes() > 0` 构造 mask，屏蔽 dummy graph
3. 把所有 reactant graph embedding 相加
4. 对 product slot 做同样编码
5. 拼接 reactant / product 的反应级表示，进入 MLP predictor

### 为什么输出是 2 维？

因为最后的 predictor 并不是只回归一个 yield 均值，而是同时输出：

- `mean`: 预测的标准化收率均值
- `logvar`: 预测的 aleatoric uncertainty 对数方差

### 下面这个代码单元会做什么？

它会把多个 reactant slot 和 product slot 组装成顶层 `reactionMPNN`，并显式写出反应级聚合：

$$
\mathbf{r} = \sum_{i=1}^{R} m_i \, \mathrm{MPNN}(G_i^{\mathrm{reactant}}),
\qquad
\mathbf{p} = \sum_{j=1}^{P} \mathrm{MPNN}(G_j^{\mathrm{product}})
$$

$$
\mathbf{z}_{\mathrm{reaction}} = [\mathbf{r};\ \mathbf{p}]
$$

其中 `m_i` 是 dummy graph mask。



In [ ]:
# ====== Step 4: 教学版 reactionMPNN 与损失函数 ======
class TeachingReactionMPNN(nn.Module):
    def __init__(
        self,
        node_in_feats: int,
        edge_in_feats: int,
        readout_feats: int = 1024,
        predict_hidden_feats: int = 512,
        prob_dropout: float = 0.1,
    ):
        super().__init__()
        self.mpnn = TeachingMPNN(node_in_feats, edge_in_feats, readout_feats=readout_feats)
        self.predict = nn.Sequential(
            nn.Linear(readout_feats * 2, predict_hidden_feats),
            nn.PReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(predict_hidden_feats, predict_hidden_feats),
            nn.PReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(predict_hidden_feats, 2),
        )

    def forward(self, rmols, pmols, return_trace: bool = False):
        reactant_mask = torch.stack([mol.batch_num_nodes() > 0 for mol in rmols]).unsqueeze(2).float()
        reactant_graph_stack = torch.stack([self.mpnn(mol) for mol in rmols])
        product_graph_stack = torch.stack([self.mpnn(mol) for mol in pmols])

        reactant_graph_feats = torch.sum(reactant_graph_stack * reactant_mask, dim=0)
        product_graph_feats = torch.sum(product_graph_stack, dim=0)
        concat_feats = torch.cat([reactant_graph_feats, product_graph_feats], dim=1)
        out = self.predict(concat_feats)

        if return_trace:
            trace = {
                'reactant_mask': reactant_mask.squeeze(2).int(),
                'reactant_graph_stack': tuple(reactant_graph_stack.shape),
                'product_graph_stack': tuple(product_graph_stack.shape),
                'reactant_graph_feats': tuple(reactant_graph_feats.shape),
                'product_graph_feats': tuple(product_graph_feats.shape),
                'concat_feats': tuple(concat_feats.shape),
                'predict_out': tuple(out.shape),
            }
            return out[:, 0], out[:, 1], trace
        return out[:, 0], out[:, 1]


def heteroscedastic_yield_loss(pred, logvar, labels, uncertainty_weight: float = 0.1):
    pointwise_mse = (pred - labels) ** 2
    return (1 - uncertainty_weight) * pointwise_mse.mean() + uncertainty_weight * (pointwise_mse * torch.exp(-logvar) + logvar).mean()

## 5. 跑一次前向推理与训练损失

### 为什么要同时看 `pred`、`logvar` 和 loss？

因为 YieldMPNN 的训练目标不是普通的 MSE。原仓库把两部分结合起来：

- 主任务：预测标准化后的 yield 均值
- 不确定性分支：学习每个样本的噪声强度 `logvar`

对应到代码里的损失近似可以写成：

$$
\mathcal{L} = (1-\lambda)\, \mathrm{MSE}(\hat{y}, y)
 + \lambda\, \mathbb{E}\!\left[(\hat{y}-y)^2 e^{-\log\sigma^2} + \log\sigma^2\right]
$$

其中默认 `\lambda = 0.1`。

### 下面这个代码单元会做什么？

它会在一个真实 batch 上同时打印 `pred`、`logvar` 和 heteroscedastic loss，让读者看到 YieldMPNN 不是普通单头回归，而是“双输出 + 不确定性加权损失”的训练目标。



In [ ]:
# ====== Step 5: 完整前向推理与损失计算 ======
torch.manual_seed(7)
model = TeachingReactionMPNN(node_in_feats=node_dim, edge_in_feats=edge_dim)
pred, logvar, reaction_trace = model(rmols, pmols, return_trace=True)

subset_indices = np.asarray(subset.indices)
train_y = dataset.yld[subset_indices]
train_y_mean = float(np.mean(train_y))
train_y_std = float(np.std(train_y))
labels_norm = (labels - train_y_mean) / train_y_std
loss = heteroscedastic_yield_loss(pred, logvar, labels_norm)

print('pred shape =', tuple(pred.shape))
print('logvar shape =', tuple(logvar.shape))
print('normalized labels =', labels_norm.numpy())
print('loss =', float(loss))
print('reaction trace:')
for key, value in reaction_trace.items():
    print(f'- {key}: {value}')

## 6. 演示 MC dropout 不确定性估计

### 核心思想

YieldMPNN 在推理阶段并不只跑一次前向，而是：

1. 先把模型切到 `eval()`
2. 再手动打开 Dropout 层
3. 对同一个 batch 做多次随机前向

然后把不确定性拆成：

- epistemic uncertainty：多次 `mean` 预测之间的方差
- aleatoric uncertainty：多次 `exp(logvar)` 的平均值

### 源码对应

- 文件：`util.py`
- 函数：`MC_dropout()`
- 文件：`model.py`
- 函数：`inference()`

### 下面这个代码单元会做什么？

它会重复执行多次随机前向，并把预测拆成 epistemic / aleatoric 两类不确定性。对应公式是：

$$
\sigma_{\mathrm{epi}}^2 = \mathrm{Var}\bigl(\hat{y}^{(1)}, \ldots, \hat{y}^{(T)}\bigr),
\qquad
\sigma_{\mathrm{ale}}^2 = \frac{1}{T} \sum_{t=1}^{T} \exp\bigl(\log \sigma_t^2\bigr)
$$

下面这格代码会把这两个量都打印出来。



In [ ]:
# ====== Step 6: MC dropout 多次前向与不确定性分解 ======
torch.manual_seed(7)
model.eval()
MC_dropout(model)

mean_draws = []
var_draws = []
with torch.no_grad():
    for _ in range(5):
        mean, logvar = model(rmols, pmols)
        mean_draws.append(mean.detach().cpu().numpy())
        var_draws.append(torch.exp(logvar).detach().cpu().numpy())

mean_draws = np.stack(mean_draws)
var_draws = np.stack(var_draws)

pred_mean = mean_draws.mean(axis=0) * train_y_std + train_y_mean
pred_epistemic = mean_draws.var(axis=0) * (train_y_std ** 2)
pred_aleatoric = var_draws.mean(axis=0) * (train_y_std ** 2)

print('mean_draws shape =', mean_draws.shape)
print('var_draws shape =', var_draws.shape)
print('denormalized predicted yield =', np.round(pred_mean, 4))
print('epistemic uncertainty =', np.round(pred_epistemic, 6))
print('aleatoric uncertainty =', np.round(pred_aleatoric, 6))

## 7. 与原仓库类实现对照

### 为什么要对照？

教学版代码的价值在于更容易理解，但仍然要确认它和原始工程的模块层次保持一致。这里不比较数值是否逐项相等，而是比较：

- 子模块构成
- 输入输出维度
- 顶层输出语义

### 下面这个代码单元会做什么？

它会把教学版 `reactionMPNN` 和原仓库类并排对照，重点检查 predictor 结构和输出张量语义，而不是要求随机初始化下的数值逐项完全相等。



In [ ]:
# ====== Step 7: 与原仓库 reactionMPNN 对照 ======
source_model = SourceReactionMPNN(node_in_feats=node_dim, edge_in_feats=edge_dim)
source_pred, source_logvar = source_model(rmols, pmols)

print('教学版输出 shape =', tuple(pred.shape), tuple(logvar.shape))
print('原仓库输出 shape =', tuple(source_pred.shape), tuple(source_logvar.shape))
print('教学版 predictor =')
print(model.predict)
print('\n原仓库 predictor =')
print(source_model.predict)

## 8. 总结

本 notebook 完成了以下工作：

1. 直接复用了原仓库的数据层，构造出与训练脚本兼容的 batch。
2. 把 `model.py` 中的 `MPNN` 和 `reactionMPNN` 拆成更容易观察的教学版实现。
3. 展示了 message passing、readout、reactant/product 汇聚、heteroscedastic loss 和 MC dropout 推理。
4. 用原仓库类做了结构级对照，确认顶层输出语义一致。

### 关键概念回顾

- `NNConv`：边特征条件化的消息传递算子
- `Set2Set`：从 node embedding 到 graph embedding 的顺序无关读出
- `mean + logvar`：同时预测均值和 aleatoric uncertainty
- `MC dropout`：通过多次随机前向估计 epistemic uncertainty

### 后续阅读建议

如果你要继续扩展这个教程，可以进一步补两件事：

- 在真实训练轮次上记录 MAE / RMSE / R2 的变化
- 对比不同 train size 下不确定性与误差的相关性